In [1]:
# Step 1: Import required libraries
import pandas as pd
import string

# Load your datasets
places = pd.read_csv("places.csv")
reviews = pd.read_csv("reviews.csv")

In [ ]:
places.head()

In [ ]:
reviews.head()

## Data Preprocessing

Group reviews

In [2]:
# Group reviews to summarize per place_id
reviews_grouped = (
    reviews
    .groupby("place_id", as_index=False)
    .agg({
        "review_id": "count",            # how many reviews per shop
        "text": lambda x: list(x)        # optional: keep list of review texts
    })
    .rename(columns={"review_id": "review_count"})
)
reviews_grouped.head()


,place_id,review_count,text
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...
1,ChIJ-00UHBxKDW0RT8IBRwA5KsM,5,[Excellent lovely Chinese food plus they have ...
2,ChIJ-080qABDDW0R2ud1Ol0zm9c,5,"[Safe, secure,easy launching, Beautiful and tr..."
3,ChIJ-0FfMP0_DW0R5OFDizOoHqI,5,[Service was great! Guy at the cashier and ser...
4,ChIJ-0FfMP0_DW0RFC1TAOQM1Fc,5,"[Staff and place is great, but noticed the chi..."


In [3]:
# Merge grouped reviews with places, now also including 'category'
shops_r = reviews_grouped.merge(
    places[["place_id", "name", "address", "lat", "lng", "user_ratings_total", "category"]],
    on="place_id",
    how="inner",
    validate="1:1"
)
shops_r.head()

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...,Huapai Roast meal,"Shop 2/330 Main Road, Kumeū 0810, New Zealand",-36.772129,174.545698,67.0,restaurant
1,ChIJ-00UHBxKDW0RT8IBRwA5KsM,5,[Excellent lovely Chinese food plus they have ...,Saint Heliers Takeaways,"19 Maskell Street, St Heliers, Auckland 1071, ...",-36.859269,174.861145,93.0,restaurant
2,ChIJ-080qABDDW0R2ud1Ol0zm9c,5,"[Safe, secure,easy launching, Beautiful and tr...",Alex Jenkins Memorial Reserve,"Titirangi, Auckland 0604, New Zealand",-36.962905,174.652572,13.0,park
3,ChIJ-0FfMP0_DW0R5OFDizOoHqI,5,[Service was great! Guy at the cashier and ser...,BurgerFuel Westgate,"579 Don Buck Road, Westgate, Auckland 0614, Ne...",-36.821870,174.608392,1115.0,restaurant
4,ChIJ-0FfMP0_DW0RFC1TAOQM1Fc,5,"[Staff and place is great, but noticed the chi...",Westgate Takeaways,"579 Don Buck Road, Westgate, Massey North 0614...",-36.821860,174.607987,287.0,restaurant


### Quick overview of the data

In [4]:
print("Places dataset-")
print(f"Rows: {places.shape[0]}, Columns: {places.shape[1]}")

print("Reviews dataset-")
print(f"Rows: {reviews.shape[0]}, Columns: {reviews.shape[1]}")

print("Grouped reviews dataset-")
print(f"Rows: {reviews_grouped.shape[0]}, Columns: {reviews_grouped.shape[1]}")

print("Merged dataset-")
print(f"Rows: {shops_r.shape[0]}, Columns: {shops_r.shape[1]}")

Places dataset-
Rows: 6501, Columns: 13
Reviews dataset-
Rows: 24805, Columns: 9
Grouped reviews dataset-
Rows: 5508, Columns: 3
Merged dataset-
Rows: 5508, Columns: 9


**Check for missingness**

In [5]:
# Count missing values in each column
shops_r.isna().sum()


place_id              0
review_count          0
text                  0
name                  0
address               0
lat                   0
lng                   0
user_ratings_total    2
category              0
dtype: int64

**Check for duplicates**

In [6]:
# Verify one unique row per place
duplicate_places = shops_r["place_id"].duplicated().sum()
print(f"Duplicate place_ids: {duplicate_places}")


Duplicate place_ids: 0


In [7]:
#print("Max reviews seen:", shops_r["review_count"].max())
#assert shops_r["review_count"].max() <= 5, "Unexpected: a place has >5 reviews"


In [8]:
# Find places with >5 reviews
shops_r[shops_r["review_count"] > 5][["place_id", "name", "review_count"]]


,place_id,name,review_count
237,ChIJ1bK-dBhHDW0RE7wELDD6bPI,KFC,6
799,ChIJ7xnZXKdPDW0RFJEDMXgNN84,Sushi Time,6
1176,ChIJCVSo8DlNDW0RJHwR9Oef5m4,Hong Yuan Flat Bush,6
1842,ChIJJyUY-CtMDW0RUF8IBWA8vmU,KFC,7
2478,ChIJRXrk8StJDW0RqUz1eWIHIDU,Goode Brothers,6
2616,ChIJT5IiTwA5DW0RCSKENlno9ZY,Lazeez kebabs&burgers,6
2695,ChIJU4Wcf-VJDW0R6L38Gukj5dY,Kohi Fresh Fish & Takeaways,6
2723,ChIJUWwDwd4VDW0RYW_ydKl9u6U,Bombay Delights Indian & Fusion Cuisine,6
2805,ChIJVV8JMMhNDW0RWujcjSX0qt8,Burger King Papatoetoe,6
2831,ChIJVeL5rrBTDW0RRB50-nC9WA4,Papas Chicken,6


In [9]:
# Look at the actual review texts for one
pid = shops_r.loc[shops_r["review_count"] > 5, "place_id"].iloc[0]
for idx, review in enumerate(shops_r.loc[shops_r["place_id"] == pid, "text"].values[0], start=1):
    print(f"{idx}. {review}\n")


1. Terrible experience tonight drive through all round. The lady at the drive through was rushing us through our order. So much so, when we got to the window, she had split our order of two meals into two separate orders. When she gave us the bag of food, she shoved it so much the bag ripped and I had to cradle it to get it into the car without the food falling out. When we got home, the drinks (sprite) were flat and not cold, the zinger burger box wasn’t even in a box and the burger was almost coming out of the packaging. Half the meat was missing off my drumstick, and the meat remaining had no coating. The chips were soggy, but I’ll get them a star as they did put extra seasoning on. Our chicken, bun and burger was all luke warm so we had to put it in the microwave to heat it up. The only thing that was good was the potato and gravy. All in all, extremely disappointing and overpriced meal and will not be returning.

2. wanted to provide feedback about my recent visit to KFC Pt Chev. 

### Text Cleaning and Preprocessing

In [10]:
import re

def clean_review_text(text):
    """Normalize review text for NLP."""
    text = str(text).lower()                          # lowercase
    text = re.sub(r"https?://\S+|www\.\S+", " ", text) # remove URLs
    text = re.sub(r"@\w+", " ", text)                  # remove mentions
    text = re.sub(r"#\w+", " ", text)                  # remove hashtags
    text = re.sub(r"[^\w\s]", " ", text)               # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()           # normalize spaces
    return text

# Create a new column with cleaned reviews for each shop
shops_r["clean_texts"] = shops_r["text"].apply(lambda reviews: [clean_review_text(r) for r in reviews])
shops_r.head()


,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...,Huapai Roast meal,"Shop 2/330 Main Road, Kumeū 0810, New Zealand",-36.772129,174.545698,67.0,restaurant,[we had a medium roast lamb and roast beef din...
1,ChIJ-00UHBxKDW0RT8IBRwA5KsM,5,[Excellent lovely Chinese food plus they have ...,Saint Heliers Takeaways,"19 Maskell Street, St Heliers, Auckland 1071, ...",-36.859269,174.861145,93.0,restaurant,[excellent lovely chinese food plus they have ...
2,ChIJ-080qABDDW0R2ud1Ol0zm9c,5,"[Safe, secure,easy launching, Beautiful and tr...",Alex Jenkins Memorial Reserve,"Titirangi, Auckland 0604, New Zealand",-36.962905,174.652572,13.0,park,"[safe secure easy launching, beautiful and tra..."
3,ChIJ-0FfMP0_DW0R5OFDizOoHqI,5,[Service was great! Guy at the cashier and ser...,BurgerFuel Westgate,"579 Don Buck Road, Westgate, Auckland 0614, Ne...",-36.821870,174.608392,1115.0,restaurant,[service was great guy at the cashier and serv...
4,ChIJ-0FfMP0_DW0RFC1TAOQM1Fc,5,"[Staff and place is great, but noticed the chi...",Westgate Takeaways,"579 Don Buck Road, Westgate, Massey North 0614...",-36.821860,174.607987,287.0,restaurant,[staff and place is great but noticed the chic...


In [11]:
shops_r.head()

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...,Huapai Roast meal,"Shop 2/330 Main Road, Kumeū 0810, New Zealand",-36.772129,174.545698,67.0,restaurant,[we had a medium roast lamb and roast beef din...
1,ChIJ-00UHBxKDW0RT8IBRwA5KsM,5,[Excellent lovely Chinese food plus they have ...,Saint Heliers Takeaways,"19 Maskell Street, St Heliers, Auckland 1071, ...",-36.859269,174.861145,93.0,restaurant,[excellent lovely chinese food plus they have ...
2,ChIJ-080qABDDW0R2ud1Ol0zm9c,5,"[Safe, secure,easy launching, Beautiful and tr...",Alex Jenkins Memorial Reserve,"Titirangi, Auckland 0604, New Zealand",-36.962905,174.652572,13.0,park,"[safe secure easy launching, beautiful and tra..."
3,ChIJ-0FfMP0_DW0R5OFDizOoHqI,5,[Service was great! Guy at the cashier and ser...,BurgerFuel Westgate,"579 Don Buck Road, Westgate, Auckland 0614, Ne...",-36.821870,174.608392,1115.0,restaurant,[service was great guy at the cashier and serv...
4,ChIJ-0FfMP0_DW0RFC1TAOQM1Fc,5,"[Staff and place is great, but noticed the chi...",Westgate Takeaways,"579 Don Buck Road, Westgate, Massey North 0614...",-36.821860,174.607987,287.0,restaurant,[staff and place is great but noticed the chic...


### Tokenisation

In [12]:
import re

# A basic stopword list (can be expanded later)
STOPWORDS = set("""
a about above after again against all am an and any are as at be because been before being below
between both but by can did do does doing down during each few for from further had has have having
he her here hers herself him himself his how i if in into is it its itself just me more most my
myself no nor not of off on once only or other our ours ourselves out over own same she should so
some such than that the their theirs them themselves then there these they this those through to too
under until up very was we were what when where which while who whom why will with you your yours
yourself yourselves
""".split())

def tokenize_and_remove_stopwords(text):
    """Split text into tokens and remove common stopwords."""
    tokens = re.findall(r"[a-z']+", text)  # words only
    tokens = [t for t in tokens if t not in STOPWORDS]
    return tokens


Clean empty reviews

In [13]:
# Remove shops where any review text in the list is empty or NaN
shops_r["clean_texts"] = shops_r["clean_texts"].apply(
    lambda reviews: [r for r in reviews if r.strip() not in ("", "nan")]
)

# Drop rows where the resulting list is empty (no valid reviews left)
shops_r = shops_r[shops_r["clean_texts"].apply(len) > 0].copy()


In [14]:
# Apply to each review in every shop
shops_r["tokens"] = shops_r["clean_texts"].apply(
    lambda reviews: [tokenize_and_remove_stopwords(r) for r in reviews]
)

shops_r.head(1)


,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...,Huapai Roast meal,"Shop 2/330 Main Road, Kumeū 0810, New Zealand",-36.772129,174.545698,67.0,restaurant,[we had a medium roast lamb and roast beef din...,"[[medium, roast, lamb, roast, beef, dinner, fi..."


In [15]:
# Look at tokens for a random shop
import random
sample_tokens = random.choice(shops_r["tokens"].values)
for i, review_tokens in enumerate(sample_tokens, start=1):
    print(f"Review {i}: {review_tokens}")


Review 1: ['wanted', 'share', 'delicious', 'lamb', 'kebab', 'today', 'garlic', 'yogurt', 'sweet', 'chili', 'mayo', 'fries', 'came', 'cute', 'little', 'package', 'nice', 'touch', 'think', 'll', 'ask', 'tomato', 'sauce', 'next', 'time', 'bit', 'much', 'overall', 'really', 'enjoyable', 'experience']
Review 2: ['pleasant', 'service', 'awesome', 'portion', 'sizes', 'tasty', 'stars', 'curing', 'hangover']
Review 3: ['never', 'kebab', 'huge', 'one', 'huge', 'tasted', 'amazing', 'couldn', 't', 'finish', 'whole', 'thing', 'even', 'half', 'bigger', 'palm', 'service', 'great', 'staff', 'super', 'friendly', 'hands', 'best', 'kebab', 'auckland', 'worth']
Review 4: ['food', 'steaming', 'hot', 'super', 'tasty', 'got', 'chicken', 'kebab', 'chicken', 'chips', 'would', 'definitely', 'eat', 'guy', 'behind', 'counter', 'really', 'nice']
Review 5: ['kebab', 'sensation', 'stoddard', 'road', 'brought', 'kebabs', 'burgers', 'hot', 'chips', 'drinks', 'late', 'lunch', 'weren', 't', 'disappointed', 'food', 'grea

Drop single character tokens

In [16]:
import re

def clean_tokens(tokens):
    out = []
    for t in tokens:
        if t == "s":               # drop possessive leftovers
            continue
        if len(t) < 2:             # drop 1-char tokens
            continue
        if re.fullmatch(r"\d+", t):# drop pure numbers
            continue
        out.append(t)
    return out

# apply to each review’s token list
shops_r["tokens_clean"] = shops_r["tokens"].apply(lambda reviews: [clean_tokens(toks) for toks in reviews])
shops_r = shops_r.drop('tokens', axis=1)
shops_r.head(1)


,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...,Huapai Roast meal,"Shop 2/330 Main Road, Kumeū 0810, New Zealand",-36.772129,174.545698,67.0,restaurant,[we had a medium roast lamb and roast beef din...,"[[medium, roast, lamb, roast, beef, dinner, fi..."


**Categorizing the places and reviews**

In [17]:
# Create four different DataFrames by category
rest_df= shops_r[shops_r["category"]== "restaurant"]
park_df= shops_r[shops_r["category"]== "park"]
mall_df= shops_r[shops_r["category"]== "shopping mall"]
tour_df= shops_r[shops_r["category"]== "tourist attraction"]

test_df=rest_df.copy()


In [18]:
# Show counts for each category
print("Restaurants:", len(rest_df))
print("Parks:", len(park_df))
print("Shopping Malls:", len(mall_df))
print("Tourist Attractions:", len(tour_df))

Restaurants: 3168
Parks: 1659
Shopping Malls: 92
Tourist Attractions: 409


In [19]:
# Restaurants
rest_df.head()

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...,Huapai Roast meal,"Shop 2/330 Main Road, Kumeū 0810, New Zealand",-36.772129,174.545698,67.0,restaurant,[we had a medium roast lamb and roast beef din...,"[[medium, roast, lamb, roast, beef, dinner, fi..."
1,ChIJ-00UHBxKDW0RT8IBRwA5KsM,5,[Excellent lovely Chinese food plus they have ...,Saint Heliers Takeaways,"19 Maskell Street, St Heliers, Auckland 1071, ...",-36.859269,174.861145,93.0,restaurant,[excellent lovely chinese food plus they have ...,"[[excellent, lovely, chinese, food, plus, grea..."
3,ChIJ-0FfMP0_DW0R5OFDizOoHqI,5,[Service was great! Guy at the cashier and ser...,BurgerFuel Westgate,"579 Don Buck Road, Westgate, Auckland 0614, Ne...",-36.821870,174.608392,1115.0,restaurant,[service was great guy at the cashier and serv...,"[[service, great, guy, cashier, serving, food,..."
4,ChIJ-0FfMP0_DW0RFC1TAOQM1Fc,5,"[Staff and place is great, but noticed the chi...",Westgate Takeaways,"579 Don Buck Road, Westgate, Massey North 0614...",-36.821860,174.607987,287.0,restaurant,[staff and place is great but noticed the chic...,"[[staff, place, great, noticed, chicken, fried..."
5,ChIJ-0FfMP0_DW0RnUqooQn0JB8,5,[Went late hours but absolutely amazing servic...,Subway - Westgate,"579 Don Buck Road, Westgate, Auckland 0614, Ne...",-36.821944,174.608023,340.0,restaurant,[went late hours but absolutely amazing servic...,"[[went, late, hours, absolutely, amazing, serv..."


In [20]:
# Parks
park_df.head()

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean
2,ChIJ-080qABDDW0R2ud1Ol0zm9c,5,"[Safe, secure,easy launching, Beautiful and tr...",Alex Jenkins Memorial Reserve,"Titirangi, Auckland 0604, New Zealand",-36.962905,174.652572,13.0,park,"[safe secure easy launching, beautiful and tra...","[[safe, secure, easy, launching], [beautiful, ..."
8,ChIJ-18TuStHDW0Rh6flFP7K_SA,2,[Looks all jungle now. the Daldy Street Commun...,Daldy Street Community Gardens and For The Lov...,"Daldy Street, Auckland Central, Auckland 1010,...",-36.842779,174.754962,2.0,park,[looks all jungle now the daldy street communi...,"[[looks, jungle, now, daldy, street, community..."
9,ChIJ-1IkrM85DW0Ru9t--hWOEdw,5,[My favourite place to take my little one for ...,Taharoto Park,"13 Taharoto Road, Takapuna, Auckland 0622, New...",-36.788565,174.760503,63.0,park,[my favourite place to take my little one for ...,"[[favourite, place, take, little, one, quick, ..."
12,ChIJ-1xcOwA7DW0RbE_4woT8gl8,1,[I had an awesome time at the Albany Marathon....,Albany Marathon,"North Shore, Albany, Auckland 0632, New Zealand",-36.726957,174.708765,1.0,park,[i had an awesome time at the albany marathon ...,"[[awesome, time, albany, marathon, early, morn..."
13,ChIJ-1yv9FFLDW0R4Y3w5Ifm-6o,5,"[Nice staff, helpful, professional and knowled...",STIHL SHOP Howick - Open 7 days,"Parking at rear of shop 102 Picton Street, How...",-36.894663,174.932763,31.0,park,[nice staff helpful professional and knowledab...,"[[nice, staff, helpful, professional, knowleda..."


In [21]:
# Malls
mall_df.head()

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean
107,ChIJ0-4gpPJGDW0RUY3nD0-Pn1k,5,[This is a very nice center. There are not too...,Tulja Centre,"190 Stoddard Road, Wesley, Auckland 1041, New ...",-36.900680,174.721294,42.0,shopping mall,[this is a very nice center there are not too ...,"[[nice, center, many, businesses, couple, time..."
115,ChIJ04jFZFS1cm0RjCaU6uIWLeo,5,[Dropped by this lovely & relatively new small...,Pohutukawa Coast Shopping Centre,"129 Beachlands Road, Beachlands 2018, New Zealand",-36.889547,175.010820,9.0,shopping mall,[dropped by this lovely relatively new small s...,"[[dropped, lovely, relatively, new, small, sho..."
178,ChIJ0wsWpn87DW0RHk-W1y2FcQ4,5,[Has everything you need for a quick feed and ...,Dairy Flat Motorway Centre,"Dairy Flat 0794, New Zealand",-36.659888,174.666403,9.0,shopping mall,[has everything you need for a quick feed and ...,"[[everything, need, quick, feed, coffee, massi..."
312,ChIJ2YxMszFLDW0RXc9zIlvgsVI,5,"[Avoid the Burger Fuel, worst service ever. 30...",Kentigern Plaza,"102 Pakuranga Road, Pakuranga, Auckland 2010, ...",-36.910881,174.871609,53.0,shopping mall,[avoid the burger fuel worst service ever 30mi...,"[[avoid, burger, fuel, worst, service, ever, m..."
399,ChIJ3TYd0-RHDW0RjdRZUxrm61Y,5,"[At one counter, a woman who jumped in line an...",NZH 康尔佳保健品连锁,"Shop 2/10 Lorne Street, Auckland Central, Auck...",-36.850022,174.765985,16.0,shopping mall,[at one counter a woman who jumped in line ana...,"[[one, counter, woman, jumped, line, analyzed,..."


In [22]:
# Tourist Attractions
tour_df.head()

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean
11,ChIJ-1k3LYJIDW0R8p3ivQR-WbA,5,[Great place for families and dog walkers. Ben...,Campbell Fountain & Statue,"308-312 Manukau Road, Epsom, Auckland 1051, Ne...",-36.888459,174.775903,87.0,tourist attraction,[great place for families and dog walkers benc...,"[[great, place, families, dog, walkers, benche..."
66,ChIJ-XCDIlBBDW0Rybt9IcvRgog,5,[Always an interesting place to visit as I’m i...,Te Toi Uku - Crown Lynn & Clayworks Museum,"8 Ambrico Place, New Lynn, Auckland 0600, New ...",-36.911109,174.680283,71.0,tourist attraction,[always an interesting place to visit as i m i...,"[[always, interesting, place, visit, intereste..."
73,ChIJ-ZvaR5FJDW0RImTC-QQrWYI,5,[Perfect place for an hour in nature amongst r...,St Johns Bush Walk,"133 Gowing Drive, Meadowbank, Auckland 1072, N...",-36.870544,174.841576,99.0,tourist attraction,[perfect place for an hour in nature amongst r...,"[[perfect, place, hour, nature, amongst, resid..."
75,ChIJ-_KMDFo4DW0ROFkm2fVfLBk,5,[Amazing sculptures and artwork. Darryl has an...,FAGENCE ART - Sculpture Garden & Gallery,"18 Paruru Avenue, Northcote, Auckland 0627, Ne...",-36.806137,174.737953,5.0,tourist attraction,[amazing sculptures and artwork darryl has an ...,"[[amazing, sculptures, artwork, darryl, great,..."
81,ChIJ-b4gbHetcm0Rr7M35V1_js8,5,[My dad flew a tiger moth back when he was 19 ...,New Zealand Warbirds Association,"824 Harvard Lane, Ardmore 2582, New Zealand",-37.032866,174.976206,368.0,tourist attraction,[my dad flew a tiger moth back when he was 19 ...,"[[dad, flew, tiger, moth, back, now, real, air..."


### Keyword extraction

In [23]:
# Define keywords
KEYWORDS = {
    "restaurant": {"clean", "cozy", "spacious", "crowded", "busy", "quiet", "noisy", "friendly", "rude",
        "welcoming", "attentive", "professional", "efficient", "slow", "quick", "prompt", "helpful",
        "approachable", "hygienic", "dirty", "tidy", "spotless", "comfortable", "uncomfortable", "overpriced",
        "expensive", "cheap", "affordable", "reasonable", "value", "worthwhile", "overrated", "excellent",
        "great", "amazing", "lovely", "fantastic", "outstanding", "average", "decent", "disappointing",
        "terrible", "awful", "poor", "superb", "nice", "pleasant", "beautiful", "gorgeous", "atmospheric",
        "decorated", "modern", "traditional", "inviting", "stylish", "safe", "unsafe"},
    "park": {"green", "lush", "clean", "peaceful", "quiet", "serene", "tranquil", "calm", "relaxing",
        "spacious", "open", "crowded", "busy", "safe", "unsafe", "family-friendly", "child-friendly",
        "pet-friendly", "dog-friendly", "accessible", "inclusive", "welcoming", "tidy", "hygienic", "dirty",
        "polluted", "beautiful", "scenic", "picturesque", "refreshing", "natural", "breezy", "shady",
        "sunny", "greenery", "landscaped", "maintained", "unkept", "secure", "unsafe", "peaceful",
        "quiet", "relaxing", "recreational", "sporty", "vibrant", "energetic", "playful", "safe",
        "clean", "refreshing", "lovely"},
    "shopping mall": {"spacious", "crowded", "busy", "quiet", "safe", "secure", "unsafe", "modern", "stylish",
        "clean", "tidy", "dirty", "hygienic", "accessible", "inclusive", "welcoming", "organized", "chaotic",
        "bright", "well-lit", "dark", "confusing", "easy", "navigable", "sprawling", "compact", "big",
        "huge", "small", "cramped", "air-conditioned", "comfortable", "uncomfortable", "overpriced", "expensive",
        "affordable", "cheap", "reasonable", "family-friendly", "child-friendly", "crowded", "popular", "busy",
        "trendy", "upscale", "luxury", "basic", "ordinary", "modernized", "outdated", "stylish", "inviting",
        "safe", "clean", "friendly"},
    "tourist attraction": {"historic", "ancient", "modern", "beautiful", "gorgeous", "scenic", "picturesque", "breathtaking",
        "majestic", "grand", "iconic", "famous", "popular", "crowded", "busy", "peaceful", "quiet", "serene",
        "clean", "tidy", "dirty", "unsafe", "safe", "secure", "accessible", "welcoming", "inclusive", "touristy",
        "authentic", "cultural", "traditional", "vibrant", "colorful", "energetic", "spiritual", "sacred",
        "artistic", "creative", "inspiring", "memorable", "remarkable", "unique", "extraordinary", "ordinary",
        "overrated", "expensive", "affordable", "reasonable", "educational", "informative", "guided", "interactive",
        "family-friendly", "child-friendly", "adventurous", "photogenic"}
}

In [24]:
def add_keywords(df, category):
    vocab = KEYWORDS[category]
    df = df.copy()
    df["keywords"] = df["tokens_clean"].apply(
        lambda reviews: sorted({w for toks in reviews for w in toks if w in vocab})
    )
    return df


In [25]:
rest_df = add_keywords(rest_df, "restaurant")
park_df = add_keywords(park_df, "park")
mall_df = add_keywords(mall_df, "shopping mall")
tour_df = add_keywords(tour_df, "tourist attraction")

test_df = add_keywords(test_df, "restaurant")

In [26]:
test_df[["place_id","name","keywords"]].head()

,place_id,name,keywords
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,Huapai Roast meal,"[decent, friendly, nice, prompt]"
1,ChIJ-00UHBxKDW0RT8IBRwA5KsM,Saint Heliers Takeaways,"[decent, excellent, expensive, great, lovely, ..."
3,ChIJ-0FfMP0_DW0R5OFDizOoHqI,BurgerFuel Westgate,"[friendly, great, nice, terrible]"
4,ChIJ-0FfMP0_DW0RFC1TAOQM1Fc,Westgate Takeaways,"[clean, friendly, great, lovely]"
5,ChIJ-0FfMP0_DW0RnUqooQn0JB8,Subway - Westgate,"[amazing, clean, expensive, friendly, great, h..."


## Sentiment analysis

In [27]:
!pip install vaderSentiment


  Using cached vaderSentiment-3.3.2-py2.py3-none-any.whl.metadata (572 bytes)
Using cached vaderSentiment-3.3.2-py2.py3-none-any.whl (125 kB)


In [28]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# initialize analyzer
analyzer = SentimentIntensityAnalyzer()


In [29]:
def score_keywords(keywords):
    if isinstance(keywords, str):
        keywords = [keywords]
    elif not isinstance(keywords, list):
        return []

    return [analyzer.polarity_scores(kw)["compound"] for kw in keywords if isinstance(kw, str)]


In [30]:
test_df["kw_scores"] = test_df["keywords"].apply(score_keywords)


In [31]:
# peek one row
test_df[["place_id","name","kw_scores"]].head(1)

,place_id,name,kw_scores
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,Huapai Roast meal,"[0.0, 0.4939, 0.4215, 0.0]"


In [32]:
test_df["keywords"].iloc[0]  # This should be a list of lists, not a long string

['decent', 'friendly', 'nice', 'prompt']

In [33]:
# Save to CSV
test_df.to_csv("test_df_with_keyword_scores.csv", index=False)


In [34]:
def aggregate_sentiment(kw_scores):
    scores = []
    for score in kw_scores:
        if isinstance(score, list):
            scores.extend(score)
        elif isinstance(score, (int, float)):
            scores.append(score)

    if not scores:
        return pd.Series([0.0, 0.0, 0.0, "neutral"], 
                         index=["avg_sentiment", "pct_positive", "pct_negative", "sentiment_label"])

    avg = sum(scores) / len(scores)
    pos = sum(1 for s in scores if s > 0.05) / len(scores)
    neg = sum(1 for s in scores if s < -0.05) / len(scores)

    if pos > neg:
        label = "positive"
    elif neg > pos:
        label = "negative"
    else:
        label = "neutral"

    return pd.Series([avg, pos, neg, label],
                     index=["avg_sentiment", "pct_positive", "pct_negative", "sentiment_label"])


In [35]:
test_df[["avg_sentiment", "pct_positive", "pct_negative", "sentiment_label"]] = (
    test_df["kw_scores"].apply(aggregate_sentiment)
)

# Repeat similarly for park_df, mall_df, tour_df if needed


In [36]:
test_df[["place_id","name","kw_scores","avg_sentiment", "pct_positive", "pct_negative", "sentiment_label"]].head(1)

,place_id,name,kw_scores,avg_sentiment,pct_positive,pct_negative,sentiment_label
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,Huapai Roast meal,"[0.0, 0.4939, 0.4215, 0.0]",0.22885,0.5,0.0,positive


#### SAMPLE TESTING

In [37]:
import random

random.seed(42)
# Randomly select 30 unique place_ids
sample_ids = random.sample(list(test_df["place_id"].dropna().unique()), 30)
sample_df = test_df[test_df["place_id"].isin(sample_ids)]

# Save to Excel
sample_df.to_excel("sample_place_ids_sentiment.xlsx", index=False)


#### TEXTBLOB

In [39]:
from textblob import TextBlob

In [40]:
def get_textblob_sentiment(texts):
    if not isinstance(texts, list):
        return []
    return [TextBlob(review).sentiment.polarity for review in texts]


In [41]:
sample_df["tb_scores"] = sample_df["clean_texts"].apply(get_textblob_sentiment)

C:\Users\Shubham\AppData\Local\Temp\ipykernel_15348\1082498390.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_df["tb_scores"] = sample_df["clean_texts"].apply(get_textblob_sentiment)


In [43]:
def aggregate_textblob_sentiment(tb_scores):
    if not isinstance(tb_scores, list) or len(tb_scores) == 0:
        return pd.Series([0.0, 0.0, 0.0, "neutral"], 
                         index=["tb_avg", "tb_pos", "tb_neg", "tb_label"])

    tb_avg = sum(tb_scores) / len(tb_scores)
    pos = sum(1 for s in tb_scores if s >= 0.05) / len(tb_scores)
    neg = sum(1 for s in tb_scores if s <= -0.05) / len(tb_scores)

    if pos > neg:
        label = "positive"
    elif neg > pos:
        label = "negative"
    else:
        label = "neutral"

    return pd.Series([tb_avg, pos, neg, label], 
                     index=["tb_avg", "tb_pos", "tb_neg", "tb_label"])


In [44]:
sample_df[["tb_avg", "tb_pos", "tb_neg", "tb_label"]] = (
    sample_df["tb_scores"].apply(aggregate_textblob_sentiment)
)

# Save to Excel
sample_df.to_excel("sample_place_ids_sentiment.xlsx", index=False)

C:\Users\Shubham\AppData\Local\Temp\ipykernel_15348\2568029044.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_df[["tb_avg", "tb_pos", "tb_neg", "tb_label"]] = (
C:\Users\Shubham\AppData\Local\Temp\ipykernel_15348\2568029044.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_df[["tb_avg", "tb_pos", "tb_neg", "tb_label"]] = (
C:\Users\Shubham\AppData\Local\Temp\ipykernel_15348\2568029044.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try u

#### BERT

In [45]:
pip install transformers torch


Note: you may need to restart the kernel to use updated packages.


In [46]:
from transformers import pipeline

# Load BERT sentiment analysis pipeline
bert_classifier = pipeline("sentiment-analysis", model="siebert/sentiment-roberta-large-english")


Device set to use cpu


In [59]:
def get_bert_sentiment(cleaned_reviews):
    if not isinstance(cleaned_reviews, list):
        return pd.Series([[], 0.0, 0.0, 0.0, "neutral"],
                         index=["bert_sentiment", "bert_score", "bert_pos", "bert_neg", "bert_label"])

    labels = []
    scores = []

    for review in cleaned_reviews:
        if isinstance(review, str) and review.strip():
            pred = bert_classifier(review[:512])[0]
            label = pred["label"].lower()
            score = pred["score"]

            labels.append(label)
            if label == "positive":
                scores.append(score)
            elif label == "negative":
                scores.append(-score)

    total = len(labels)
    pos = labels.count("positive")
    neg = labels.count("negative")
    pct_pos = pos / total if total > 0 else 0.0
    pct_neg = neg / total if total > 0 else 0.0
    avg_score = round(sum(scores) / len(scores), 4) if scores else 0.0

    if pos > neg:
        overall = "positive"
    elif neg > pos:
        overall = "negative"
    else:
        overall = "neutral"

    return pd.Series([labels, avg_score, pct_pos, pct_neg, overall],
                     index=["bert_sentiment", "bert_score", "bert_pos", "bert_neg", "bert_label"])


In [64]:
sample_df[["bert_sentiment", "bert_score", "bert_pos", "bert_neg", "bert_label"]] = sample_df["clean_texts"].apply(get_bert_sentiment)


C:\Users\Shubham\AppData\Local\Temp\ipykernel_15348\2363709600.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_df[["bert_sentiment", "bert_score", "bert_pos", "bert_neg", "bert_label"]] = sample_df["clean_texts"].apply(get_bert_sentiment)


In [66]:
sample_df.head(1)

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,...,tb_scores,tb_avg,tb_pos,tb_neg,tb_label,bert_sentiment,bert_score,bert_pos,bert_neg,bert_label
171,ChIJ0e989wk7DW0RnfvTQBhLL4I,2,[Chicken pies tonight.\nStaff seemed a bit gru...,Kats Kitchen,"75 Oaktree Avenue, Browns Bay, Auckland 0630, ...",-36.722304,174.731495,2.0,restaurant,[chicken pies tonight staff seemed a bit grump...,...,"[-0.28437500000000004, 0.85]",0.282812,0.5,0.5,neutral,"[negative, positive]",-0.0002,0.5,0.5,neutral


In [67]:
# Save to Excel
sample_df.to_excel("sample_place_ids_sentiment.xlsx", index=False)

#### MODEL COMPARISON

In [ ]:
# Load the file
comp_df = pd.read_excel("manual labels for reviews.xlsx")

In [ ]:
man_agg = []
for i in range(len(comp_df)):
    labels = comp_df.loc[i, "manual_labels"].split(",")
    labels = [l.strip().lower() for l in labels]  # normalize case just in case
    counts = pd.Series(labels).value_counts()

    if (counts == counts.max()).sum() > 1:
        agg = "neutral"
    else:
        agg = counts.idxmax()

    man_agg.append(agg)

comp_df["man_agg"] = man_agg


In [ ]:
comp_df.head(1)

In [ ]:
from sklearn.metrics import accuracy_score

# Compare manual labels vs each model
vader_acc = accuracy_score(comp_df["man_agg"], comp_df["sentiment_label"])
textblob_acc = accuracy_score(comp_df["man_agg"], comp_df["tb_label"])
bert_acc = accuracy_score(comp_df["man_agg"], comp_df["bert_label"])

# Print results
print("Accuracy vs Manual Labels:")
print(f"VADER Accuracy     : {vader_acc:.2f}")
print(f"TextBlob Accuracy  : {textblob_acc:.2f}")
print(f"BERT Accuracy      : {bert_acc:.2f}")


In [ ]:
comp_df['man_agg'].describe()

**Data is imbalanced. Positive counts are 24/30, so accuracy alone is misleading. Therefore classification report can provide deeper insight on model selection**

In [ ]:
from sklearn.metrics import classification_report

print("VADER:")
print(classification_report(comp_df["man_agg"], comp_df["sentiment_label"]))

print("TextBlob:")
print(classification_report(comp_df["man_agg"], comp_df["tb_label"]))

print("BERT:")
print(classification_report(comp_df["man_agg"], comp_df["bert_label"]))


#### Classification Reports

**VADER**:
- **Strengths**: High accuracy on positive sentiment (Precision = 0.89, Recall = 1.00)
- **Weaknesses**: Very poor recall on negative reviews (Recall = 0.20), meaning it often misses them.
- **Neutral detection**: Correctly detected the only neutral sample.

**TextBlob**:
- **Strengths**: High precision and recall across all categories. Neutral and positive classes were perfectly predicted.
- **Weaknesses**: Slight underperformance on negative recall (0.80), but overall very strong.

**BERT**:
- **Strengths**: Most balanced performance, excellent at detecting both positive and negative reviews.
- **Weaknesses**: Missed just one positive review (Recall = 0.96).
- **Neutral detection**: Perfect.

---

#### Interpretation

- **Accuracy alone is misleading** due to class imbalance (80% positive).
- **VADER** struggles with subtle or implied negativity.
- **TextBlob** offers excellent performance with simple implementation.
- **BERT** is the most robust and balanced model — ideal for production-level sentiment tasks.

---

#### ✅ Final Model Selection: BERT

Based on the evaluation of sentiment models (VADER, TextBlob, and BERT) against manual labels:

- **BERT** provided the **most balanced and accurate** performance across all sentiment classes.
- It achieved:
  - **100% recall for negative and neutral** sentiments
  - **96% recall for positive** sentiments
  - **Overall accuracy of 97%**, matching TextBlob but with **stronger performance on the negative class**.

---

In [68]:
rest_df[["bert_sentiment", "bert_score", "bert_pos", "bert_neg", "bert_label"]] = rest_df["clean_texts"].apply(get_bert_sentiment)

# Save to Excel
rest_df.to_excel("restaurants_sentiment.xlsx", index=False)

In [69]:
rest_df.head(1)

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean,keywords,bert_sentiment,bert_score,bert_pos,bert_neg,bert_label
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...,Huapai Roast meal,"Shop 2/330 Main Road, Kumeū 0810, New Zealand",-36.772129,174.545698,67.0,restaurant,[we had a medium roast lamb and roast beef din...,"[[medium, roast, lamb, roast, beef, dinner, fi...","[decent, friendly, nice, prompt]","[positive, positive, positive, positive, posit...",0.9989,1.0,0.0,positive


In [70]:
park_df[["bert_sentiment", "bert_score", "bert_pos", "bert_neg", "bert_label"]] = park_df["clean_texts"].apply(get_bert_sentiment)
park_df.head(1)

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean,keywords,bert_sentiment,bert_score,bert_pos,bert_neg,bert_label
2,ChIJ-080qABDDW0R2ud1Ol0zm9c,5,"[Safe, secure,easy launching, Beautiful and tr...",Alex Jenkins Memorial Reserve,"Titirangi, Auckland 0604, New Zealand",-36.962905,174.652572,13.0,park,"[safe secure easy launching, beautiful and tra...","[[safe, secure, easy, launching], [beautiful, ...","[beautiful, quiet, safe, secure, tranquil]","[positive, positive, positive, positive, posit...",0.9987,1.0,0.0,positive


In [71]:
# Save to Excel
park_df.to_excel("parks_sentiment.xlsx", index=False)

In [72]:
mall_df[["bert_sentiment", "bert_score", "bert_pos", "bert_neg", "bert_label"]] = mall_df["clean_texts"].apply(get_bert_sentiment)
mall_df.head(1)

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean,keywords,bert_sentiment,bert_score,bert_pos,bert_neg,bert_label
107,ChIJ0-4gpPJGDW0RUY3nD0-Pn1k,5,[This is a very nice center. There are not too...,Tulja Centre,"190 Stoddard Road, Wesley, Auckland 1041, New ...",-36.90068,174.721294,42.0,shopping mall,[this is a very nice center there are not too ...,"[[nice, center, many, businesses, couple, time...","[big, small]","[positive, positive, negative, positive, negat...",0.1993,0.6,0.4,positive


In [73]:
# Save to Excel
mall_df.to_excel("malls_sentiment.xlsx", index=False)

In [74]:
tour_df[["bert_sentiment", "bert_score", "bert_pos", "bert_neg", "bert_label"]] = tour_df["clean_texts"].apply(get_bert_sentiment)
tour_df.head(1)

,place_id,review_count,text,name,address,lat,lng,user_ratings_total,category,clean_texts,tokens_clean,keywords,bert_sentiment,bert_score,bert_pos,bert_neg,bert_label
11,ChIJ-1k3LYJIDW0R8p3ivQR-WbA,5,[Great place for families and dog walkers. Ben...,Campbell Fountain & Statue,"308-312 Manukau Road, Epsom, Auckland 1051, Ne...",-36.888459,174.775903,87.0,tourist attraction,[great place for families and dog walkers benc...,"[[great, place, families, dog, walkers, benche...",[crowded],"[positive, positive, positive, positive, posit...",0.9989,1.0,0.0,positive


In [75]:
# Save to Excel
tour_df.to_excel("tourist_sentiment.xlsx", index=False)

In [76]:
# Concatenate all category dataframe
final_sent = pd.concat([rest_df, mall_df, tour_df, park_df], ignore_index=True)
final_sent.to_excel("sentiment_merged.xlsx", index=False)

### Aspect-Based Sentiment Analysis (ABSA)

In [11]:
# Install required libraries
!pip install transformers pandas openpyxl hf_xet --quiet 


In [12]:
from transformers import pipeline

# Use a small and public model (multilingual, fast)
sentiment_model = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")


Device set to use cpu


In [14]:
import ast

# Load your file
rest_aspect = pd.read_excel("restaurants_sentiment.xlsx")
# Convert stringified list to actual list
rest_aspect["clean_texts"] = rest_aspect["clean_texts"].apply(ast.literal_eval)

In [ ]:
# Define mapping from model labels to numeric values
label_map = {
    "1 star": 1,
    "2 stars": 2,
    "3 stars": 3,
    "4 stars": 4,
    "5 stars": 5
}

# Function to score list of reviews
def score_reviews(review_list):
    results = sentiment_model(review_list)
    scores = [label_map[r['label']] for r in results]
    return {
        "avg_sentiment": sum(scores)/len(scores),
        "max_sentiment": max(scores),
        "min_sentiment": min(scores),
        "positive_pct": sum(1 for s in scores if s >= 4) / len(scores),
        "review_count": len(scores),
        "individual_scores": scores
    }
